# DDoS ML Detection Lab

This notebook is the cleaned replacement for the original `Untitled.ipynb` workflow. It uses the project modules under `src/` so the notebook, tests, and command-line experiment all use the same logic.

Research question: can a small ML pipeline distinguish benign traffic from DDoS flows in CSE-CIC-IDS2018 without relying on obvious leakage or random-row test splits?

## Methodology Summary

- Dataset: CSE-CIC-IDS2018 on AWS.
- Scope: binary Benign vs DDoS flow classification.
- Included DDoS labels: LOIC HTTP, HOIC, and LOIC UDP.
- Final test strategy: hold out `02-21-2018.csv` as a source-file test set.
- Leakage controls: remove identifiers, IPs, timestamps, source port, destination port, and source filename as model features.
- Preprocessing: median imputation inside scikit-learn pipelines fitted only on training data.

In [ ]:
from pathlib import Path
from types import SimpleNamespace
import os
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src.train import run_experiment


In [ ]:
data_dir = Path("data")
expected_files = [data_dir / "02-20-2018.csv", data_dir / "02-21-2018.csv"]
missing = [str(path) for path in expected_files if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing required local dataset files: " + ", ".join(missing)
    )

## Run Corrected Experiment

This run uses a capped, deterministic sample so the project is reproducible on a normal laptop. It still scans the two DDoS source CSVs and keeps the final test day separate.

In [ ]:
args = SimpleNamespace(
    data_dir="data",
    reports_dir="reports",
    holdout_file="02-21-2018.csv",
    include_files=["02-20-2018.csv", "02-21-2018.csv"],
    max_rows_per_class_per_file=10_000,
    chunksize=100_000,
    validation_size=0.25,
    seed=42,
    keep_dst_port=False,
    skip_permutation_importance=False,
    permutation_rows=5_000,
)
results = run_experiment(args)
results["best_model"]

## Results

The validation results are high, but the held-out source-file test shows poor DDoS recall. This is the main corrected finding: the original random-row style result was too optimistic.

In [ ]:
results_table = pd.read_csv("reports/model_results.csv")
cols = [
    "model",
    "split",
    "precision",
    "recall",
    "f1",
    "balanced_accuracy",
    "roc_auc",
    "pr_auc",
    "false_positive_rate",
    "threshold",
]
results_table[cols]

In [ ]:
best = results["best_model"]
results["model_results"][best]["test"]["confusion_matrix"]

## Feature Interpretation

Permutation importance is computed on the held-out data. These scores are not causal explanations; they are a small sanity check of which features affected the selected model on this specific holdout sample.

In [ ]:
importance = pd.read_csv("reports/permutation_importance.csv")
importance.head(10)

## Conclusion

The repaired project does not claim production-ready DDoS detection. It shows a more defensible research workflow and demonstrates why leakage controls and grouped holdouts matter in security ML experiments.